# 06 Tvätta data

I notebook 03, 04 och 05 hämtade vi data från Webperf, Wikidata och Chrome UX Report och sparade svaren som de kom i mappen `data/radata`. Nu gör vi om rådatan till en tydlig tabell per källa.

En tvättad tabell har en rad per sak, till exempel en rad per webbplats. Varje kolumn har ett namn som säger vad den innehåller och rätt datatyp. Saknade värden är tomma och inte dolda som text eller nollor.

I den här notebooken hämtar vi ingenting. Vi läser bara rådatan. Rådatan ändrar vi aldrig. Allt vi gör med den står i koden, så att vi kan köra om tvättningen och få samma resultat.

In [ ]:
import json
from pathlib import Path
from urllib.parse import urlsplit

import pandas as pd

RADATA = Path("..") / "data" / "radata"
TVATTAD = Path("..") / "data" / "tvattad"
TVATTAD.mkdir(parents=True, exist_ok=True)

Vi skriver en funktion som läser en fil med rådata. Alla våra filer har samma form, med uppgifter om när vi hämtade datan och själva svaret under `svar`.

In [ ]:
def las_radata(filnamn):
    return json.loads((RADATA / filnamn).read_text(encoding="utf-8"))

## Webperf

Vi läser kommuner och regioner och lägger dem i en tabell, som tidigare.

In [ ]:
kommuner_fil = las_radata("webperf-kommuner.json")
regioner_fil = las_radata("webperf-regioner.json")

kommuner = pd.json_normalize(kommuner_fil["svar"]["municipalities"])
regioner = pd.json_normalize(regioner_fil["svar"]["regions"])
kommuner["typ"] = "kommun"
regioner["typ"] = "region"

webperf = pd.concat([kommuner, regioner], ignore_index=True)

webperf.columns.tolist()

### En kolumn för varje sak

Kolumnerna `name` och `title` innehåller samma namn. Vi kontrollerar det innan vi tar bort den ena.

In [ ]:
(webperf["name"] == webperf["title"]).all()

### Tydliga kolumnnamn

Vi ger kolumnerna svenska namn utan å, ä och ö, med understreck mellan orden. Då blir de lätta att skriva i koden och det syns direkt vad de innehåller. Id-numret döper vi till `webperf_id`, så att det inte blandas ihop med andra källors id.

`rename` tar en ordbok där nyckeln är det gamla namnet och värdet det nya. Med en lista inom hakparenteser väljer vi samtidigt ut de kolumner vi vill behålla, i den ordning vi vill ha dem. Kolumnen `name` kommer inte med.

In [ ]:
webperf = webperf.rename(columns={
    "id": "webperf_id",
    "title": "namn",
    "website": "webbplats",
    "same_as": "wikidata_url",
    "ratings.overall": "betyg_totalt",
    "ratings.accessibility": "betyg_tillganglighet",
    "ratings.performance": "betyg_prestanda",
    "ratings.security": "betyg_sakerhet",
    "ratings.standards": "betyg_standard",
})

webperf = webperf[[
    "webperf_id", "namn", "typ", "webbplats", "wikidata_url",
    "betyg_totalt", "betyg_tillganglighet", "betyg_prestanda", "betyg_sakerhet", "betyg_standard",
]]

webperf.head()

### Nycklar till de andra källorna

För att koppla ihop källorna behöver vi två nycklar. Id-numret i Wikidata kopplar till Wikidata och originet kopplar till CrUX. Vi tar fram båda på samma sätt som i notebook 04 och 05.

In [ ]:
def till_origin(adress):
    delar = urlsplit(adress)
    return f"{delar.scheme}://{delar.netloc}"


webperf["qid"] = webperf["wikidata_url"].str.rsplit("/", n=1).str[-1]
webperf["origin"] = webperf["webbplats"].map(till_origin)

webperf[["namn", "webbplats", "origin", "qid"]].head()

En nyckel måste vara unik, annars kopplas en rad ihop med fel rad eller med flera. `is_unique` svarar `True` om inget värde förekommer två gånger.

In [ ]:
for kolumn in ["webperf_id", "qid", "origin"]:
    print(kolumn, webperf[kolumn].is_unique)

### Text utan överraskningar

Mellanslag före eller efter en text syns inte när vi tittar på tabellen, men gör att två namn som ser likadana ut inte är lika. `str.strip()` tar bort sådana mellanslag. Vi räknar hur många värden som skulle ändras.

In [ ]:
for kolumn in ["namn", "webbplats", "wikidata_url"]:
    andras = (webperf[kolumn] != webperf[kolumn].str.strip()).sum()
    print(kolumn, andras)

Inga värden behöver ändras här. Kontrollen är ändå värd att göra, eftersom det är ett av de vanligaste felen i data som skrivits in för hand.

### När hämtades datan?

Vi lägger till när datan hämtades som en egen kolumn. Då följer uppgiften med tabellen, även när den sparas och läses in igen.

In [ ]:
webperf["hamtad"] = pd.Timestamp(kommuner_fil["hamtad"])

webperf.dtypes

## Wikidata

Svaren från Wikidata ligger i fyra filer. Vi skriver en funktion som gör om ett svar till en tabell, ungefär som i notebook 04, och tar fram id-numret ur adressen i kolumnen `item`. Svaret om länens befolkning har ingen sådan kolumn, eftersom det kopplas via länskoden, så där hoppar funktionen över det steget.

In [ ]:
def wikidata_tabell(filnamn):
    fil = las_radata(filnamn)
    rader = []
    for traff in fil["svar"]["results"]["bindings"]:
        rader.append({namn: varde["value"] for namn, varde in traff.items()})
    tabell = pd.DataFrame(rader)
    if "item" in tabell.columns:
        tabell["qid"] = tabell["item"].str.rsplit("/", n=1).str[-1]
        tabell = tabell.drop(columns="item")
    return tabell, fil["hamtad"]


kommunkod, hamtad_wikidata = wikidata_tabell("wikidata-kommunkod.json")
lan, _ = wikidata_tabell("wikidata-lan.json")
befolkning, _ = wikidata_tabell("wikidata-befolkning.json")
lan_befolkning, _ = wikidata_tabell("wikidata-lan-befolkning.json")

kommunkod.shape, lan.shape, befolkning.shape, lan_befolkning.shape

Vi ger kolumnerna tydliga namn och rätt typ. Kommunkoden och länskoden förblir text, eftersom de kan börja med en nolla. Befolkningen blir heltal och tidpunkten ett datum. Tidpunkten ser ut som `2026-06-30T00:00:00Z`, och med `str[:10]` tar vi de första tio tecknen, som är själva datumet.

In [ ]:
lan = lan.rename(columns={"lanLabel": "lan"})

befolkning = befolkning.rename(columns={"tidpunkt": "befolkning_datum"})
befolkning["befolkning"] = befolkning["befolkning"].astype("Int64")
befolkning["befolkning_datum"] = pd.to_datetime(befolkning["befolkning_datum"].str[:10])

lan_befolkning = lan_befolkning.rename(columns={"lan_tidpunkt": "lan_befolkning_datum"})
lan_befolkning["lan_befolkning"] = lan_befolkning["lan_befolkning"].astype("Int64")
lan_befolkning["lan_befolkning_datum"] = pd.to_datetime(lan_befolkning["lan_befolkning_datum"].str[:10])

Vi slår ihop delarna till en tabell med en rad per id-nummer. Vi utgår från länen, eftersom alla 310 har ett län, och använder `how="left"` så att regionerna kommer med trots att de saknar kommunkod och befolkning.

In [ ]:
wikidata = (
    lan
    .merge(kommunkod, on="qid", how="left")
    .merge(befolkning, on="qid", how="left")
    .merge(lan_befolkning, on="lanskod", how="left")
)
wikidata = wikidata[[
    "qid", "kommunkod", "lan", "lanskod", "befolkning", "befolkning_datum",
    "lan_befolkning", "lan_befolkning_datum",
]]
wikidata["hamtad"] = pd.Timestamp(hamtad_wikidata)

wikidata.shape, wikidata["qid"].is_unique

In [ ]:
wikidata.head()

## CrUX

Rådatan från CrUX innehåller ett svar per origin, både de som fick data och de som fick statuskod 404. Vi behåller alla, så att det syns i tabellen vilka som saknar data. Annars skulle de bara försvinna.

Vi plockar ut måtten med samma sorts funktion som i notebook 05. För origin utan data blir alla mått tomma.

In [ ]:
crux_fil = las_radata("crux.json")


def plocka_ut(origin, status, svar):
    rad = {"origin": origin, "crux_status": status}
    if status != 200:
        return rad
    matt = svar["record"]["metrics"]
    period = svar["record"]["collectionPeriod"]
    for namn, kolumn in [
        ("largest_contentful_paint", "lcp_p75_ms"),
        ("interaction_to_next_paint", "inp_p75_ms"),
        ("cumulative_layout_shift", "cls_p75"),
    ]:
        if namn in matt:
            rad[kolumn] = matt[namn]["percentiles"]["p75"]
    rad["period_start"] = "{year}-{month:02d}-{day:02d}".format(**period["firstDate"])
    rad["period_slut"] = "{year}-{month:02d}-{day:02d}".format(**period["lastDate"])
    return rad


rader = [plocka_ut(origin, post["status"], post["svar"]) for origin, post in crux_fil["svar"].items()]
crux = pd.DataFrame(rader)

crux.shape

Kolumnnamnen säger nu också vilken enhet värdet har. `lcp_p75_ms` är millisekunder. CLS har ingen enhet.

Vi sätter rätt typer. LCP och INP blir heltal som kan sakna värde, CLS ett decimaltal och perioden datum.

In [ ]:
crux["lcp_p75_ms"] = crux["lcp_p75_ms"].astype("Int64")
crux["inp_p75_ms"] = crux["inp_p75_ms"].astype("Int64")
crux["cls_p75"] = pd.to_numeric(crux["cls_p75"])
crux["period_start"] = pd.to_datetime(crux["period_start"])
crux["period_slut"] = pd.to_datetime(crux["period_slut"])
crux["hamtad"] = pd.Timestamp(crux_fil["hamtad"])

crux.dtypes

In [ ]:
crux[crux["crux_status"] != 200]

`<NA>`, `NaN` och `NaT` betyder alla att värdet saknas. Pandas visar `<NA>` för de nyare typerna som `Int64`, `NaN` för decimaltal och `NaT` för datum och tider.

## Spara de tvättade tabellerna

Vi sparar tabellerna i formatet Parquet i mappen `data/tvattad`. Till skillnad från CSV kommer Parquet ihåg datatyperna. En kommunkod som `0180` förblir text och ett datum förblir ett datum när vi läser in filen igen. Vi går igenom olika sätt att lagra data i notebook 09.

In [ ]:
for namn, tabell in [("webperf", webperf), ("wikidata", wikidata), ("crux", crux)]:
    fil = TVATTAD / f"{namn}.parquet"
    tabell.to_parquet(fil, index=False)
    print(fil, tabell.shape)

`index=False` betyder att radnumren inte sparas, eftersom de inte säger något om datan.

Vi läser in en av filerna igen och kontrollerar att datatyperna har följt med.

In [ ]:
pd.read_parquet(TVATTAD / "wikidata.parquet").dtypes

## Det här vet vi nu

- Tvättning gör rådata till tabeller med en rad per sak, tydliga kolumnnamn och rätt datatyper.
- Rådatan ändras aldrig. Tvättningen är kod som går att köra om.
- Nycklar som ska koppla ihop källor måste vara unika.
- Saknade värden ska synas i tabellen och inte försvinna.
- Parquet behåller datatyperna när vi sparar.

I nästa notebook harmoniserar vi källorna och slår ihop dem till en tabell.